In [1]:
import torch
import plotly.express as px
import pandas as pd
import numpy as np
from typing import Optional, List, Tuple
import pathlib
from torch import device
from epsilon_transformers.persistence import Persister
from epsilon_transformers.process.processes import PROCESS_REGISTRY

from pandas.core.computation.ops import Op
import plotly.graph_objects as go
import plotly.express as px
from sklearn.decomposition import PCA
import pandas as pd

/opt/anaconda3/envs/epstrans311/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/opt/anaconda3/envs/epstrans311/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because a

In [2]:
checkpoint_dir = pathlib.Path("/Users/sbhandari/Documents/GitHub/epsilon-transformers/models/1layer_0.15_0.6_lr0.02_ckpt4000/epsilon-transformers/models/linear_mess3/1layer_0.15_0.6_lr0.02_ckpt4000/")
if torch.cuda.is_available():
    device = device("cuda:0")
elif torch.backends.mps.is_available():
    device = device("mps")
else:
    device = device("cpu")

persister = Persister(checkpoint_dir)
model_name="/Users/sbhandari/Documents/GitHub/epsilon-transformers/models/1layer_0.15_0.6_lr0.02_ckpt4000/epsilon-transformers/models/linear_mess3/1layer_0.15_0.6_lr0.02_ckpt4000/"
model=persister.load_final_model()
train_config = persister.load_training_config()
#model=persister.load_model("path")
model.eval()

[Persister] Found 4007 checkpoints in /Users/sbhandari/Documents/GitHub/epsilon-transformers/models/1layer_0.15_0.6_lr0.02_ckpt4000/epsilon-transformers/models/linear_mess3/1layer_0.15_0.6_lr0.02_ckpt4000


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0): TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (hook_resid_pre)

In [3]:
process_name = 'Linear_Mess3'
process_params ={
    "x": 0.15,
    "a": 0.6
}
seq_len = 10
vocab = 3
if process_name in PROCESS_REGISTRY:
    process=PROCESS_REGISTRY[process_name](**process_params)

In [ ]:
# history=process.generate_process_history(total_length=10)
# input_seq=torch.tensor([history.symbols],dtype=torch.long,device=device)
# print(input_seq)


tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], device='mps:0')


In [5]:
def get_attention_hooks(model):
    return [name for name,_ in model.named_modeules() if "attn.hook_pattern" in name]

In [6]:
def plot_attention_heatmap(
    cache,
    layer_idx:Optional[int]=None,
    head_idx: Optional[int]=None,
    tokens:Optional[list[str]]=None
):
    if layer_idx is None:
        all_layer_indices=[]
        for key in cache.keys():
            if key.endswith("attn.hook_pattern") and "blocks." in key:
                idx=int(key.split('.')[1])
                all_layer_indices.append(idx)
        unique_layers=sorted(list(set(all_layer_indices)))
        for idx in unique_layers:
            plot_attention_heatmap(cache,layer_idx=idx,head_idx=head_idx,tokens=tokens) 

    hook_name=f"blocks.{layer_idx}.attn.hook_pattern"
    if hook_name not in cache:
        print("error: hook not found in cache")
        return
    #pattern has shape [batch, n_heads, query_pos, key_pos]    
    pattern=cache[hook_name][0].detach().cpu().numpy()
    n_heads=pattern.shape[0]
    seq_len=pattern.shape[1]  

    if tokens is None:
        x_labels=[f"Key{i}" for i in range(seq_len)] 
        y_labels=[f"Query{i}" for i in range(seq_len)]
    else:
        x_labels = [f"{t} (K{i})" for i, t in enumerate(tokens)]
        y_labels = [f"{t} (Q{i})" for i, t in enumerate(tokens)]

    if head_idx is not None:
        print(f"Plotting Layer {layer_idx}, Head {head_idx}")
        fig = px.imshow(
            pattern[head_idx],
            labels=dict(x="Key (Source)", y="Query (Destination)", color="Attention"),
            x=x_labels,
            y=y_labels,
            title=f"Attention Pattern: Layer {layer_idx}, Head {head_idx}",
            color_continuous_scale="Viridis",
            range_color=[0, 1]
        )
        fig.update_layout(width=600, height=600)
    else:
        # print(f"Plotting All Heads for Layer {layer_idx}")
        fig = px.imshow(
            pattern,
            labels=dict(x="Key", y="Query", color="Attn", facet_col="Head"),
            x=x_labels,
            y=y_labels,
            facet_col=0, # Facet over the first dimension (heads)
            facet_col_wrap=min(n_heads, 4),  # Wrap after 4 heads
            facet_col_spacing=0.001,          # Reduced horizontal spacing (default ~0.02)
            facet_row_spacing=0.001,          # Reduced vertical spacing (default ~0.07)
            title=f"Attention Patterns: Layer {layer_idx} (All Heads)",
            color_continuous_scale="Viridis",
            range_color=[0, 1]
        )
        n_rows = (n_heads + 3) // 4
        fig.update_layout(height=400 * n_rows, width=650)

    fig.show()

In [14]:
history=process.generate_process_history(total_length=10)
input_seq=torch.tensor([history.symbols],dtype=torch.long,device=device)
# input_seq=torch.tensor([[0, 0, 1, 0, 1, 2, 0, 1, 0, 1]],dtype=torch.long,device=device)
print(input_seq)
logits, cache = model.run_with_cache(input_seq)
token_labels = [str(t.item()) for t in input_seq[0]]
plot_attention_heatmap(cache, layer_idx=None, head_idx=None, tokens=token_labels)

print(f"model{model_name}")

tensor([[2, 2, 2, 2, 1, 2, 1, 0, 0, 1]], device='mps:0')


error: hook not found in cache
model/Users/sbhandari/Documents/GitHub/epsilon-transformers/models/1layer_0.15_0.6_lr0.02_ckpt4000/epsilon-transformers/models/linear_mess3/1layer_0.15_0.6_lr0.02_ckpt4000/


In [21]:
token_colors = {
    0: '#1f77b4',  # Blue
    1: '#ff7f0e',  # Orange
    2: '#2ca02c'   # Green
}

In [15]:
def extract_vectors(model,layer_idx, head_idx):
    W_E=model.W_E.detach().cpu()
    W_V=model.blocks[layer_idx].attn.W_V[head_idx].detach().cpu()
    W_O=model.blocks[layer_idx].attn.W_O[head_idx].detach().cpu()
    if W_V.dim() > 2: W_V = W_V.squeeze()
    if W_O.dim() > 2: W_O = W_O.squeeze()
    print(W_E.shape, W_V.shape, W_O.shape)
    W_OV=W_V@W_O
    OV_vectors=W_E@W_OV
    return W_E, OV_vectors

In [ ]:
import torch
import numpy as np
from sklearn.decomposition import PCA
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_real_ov_vectors(model, process, num_seqs=50, seq_len=10, layer_idx=0, head_idx=0, start_pos=1):
    device = model.cfg.device
    
    if hasattr(process, 'generate_batch_gpu'):
        input_batch = process.generate_batch_gpu(batch_size=num_seqs, seq_len=seq_len, device=device)

    _, cache = model.run_with_cache(input_batch)

    resid_mid = cache[f"blocks.{layer_idx}.hook_resid_mid"]

    resid_mid_flat = resid_mid.reshape(-1, model.cfg.d_model).detach().cpu().float()
    
    pca = PCA(n_components=3)
    pca.fit(resid_mid_flat)

    mean_pca = np.dot(pca.mean_[None], pca.components_.T) # [1, d_model]
    resid_pre = cache[f"blocks.{layer_idx}.hook_resid_pre"][:, start_pos:, :]
    inputs_flat = resid_pre.reshape(-1, model.cfg.d_model).detach().cpu().float()
    v_hook = cache[f"blocks.{layer_idx}.attn.hook_v"][:, start_pos:, head_idx, :] 
    W_O = model.blocks[layer_idx].attn.W_O[head_idx].detach()
    ov_out = v_hook @ W_O 
    ov_flat = ov_out.reshape(-1, model.cfg.d_model).detach().cpu().float()
    inputs_pca = pca.transform(inputs_flat)[:, :2] + mean_pca[:, :2]
    ov_pca = pca.transform(ov_flat)[:, :2]
    labels = input_batch[:, start_pos:].cpu().flatten().tolist()

    fig = make_subplots(
        rows=1, cols=2, 
        subplot_titles=("Individual Token Instances", "Mean Vector per Token"),
        horizontal_spacing=0.1
    )
    
    # Colors for tokens
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    
    unique_tokens, counts = np.unique(labels, return_counts=True)
    sorted_indices = np.argsort(-counts)
    tokens_to_plot = unique_tokens[sorted_indices][:10]

    for i, token_id in enumerate(tokens_to_plot):
        c = colors[i % len(colors)]
        
        indices = [idx for idx, label in enumerate(labels) if label == token_id]
        if not indices: continue
        
        token_inputs = inputs_pca[indices]
        token_ovs = ov_pca[indices]
        
        # Scatter Instances
        # 1. Plot Input Points
        fig.add_trace(go.Scatter(
            x=token_inputs[:, 0], y=token_inputs[:, 1],
            mode='markers',
            marker=dict(size=5, color=c, opacity=0.4),
            name=f'T{token_id}',
            legendgroup=f'T{token_id}'
        ), row=1, col=1)
        
        # 2. Plot OV Lines (Vectors)
        # Construct line segments (Start -> End, None)
        x_lines, y_lines = [], []
        for j in range(len(token_inputs)):
            sx, sy = token_inputs[j, 0], token_inputs[j, 1]
            ex, ey = sx + token_ovs[j, 0], sy + token_ovs[j, 1]
            x_lines.extend([sx, ex, None])
            y_lines.extend([sy, ey, None])
            
        fig.add_trace(go.Scatter(
            x=x_lines, y=y_lines,
            mode='lines',
            line=dict(color=c, width=1),
            opacity=0.3,
            showlegend=False
        ), row=1, col=1)

        #Mean Vectors
        mean_input = token_inputs.mean(axis=0)
        mean_ov = token_ovs.mean(axis=0)
        
        # 1. Mean Input Dot
        fig.add_trace(go.Scatter(
            x=[mean_input[0]], y=[mean_input[1]],
            mode='markers+text',
            text=[f"T{token_id}"], textposition="bottom center",
            marker=dict(size=12, color=c, line=dict(width=1, color='black')),
            showlegend=False,
            legendgroup=f'T{token_id}'
        ), row=1, col=2)
        
        # 2. Mean OV Arrow
        fig.add_trace(go.Scatter(
            x=[mean_input[0], mean_input[0] + mean_ov[0]],
            y=[mean_input[1], mean_input[1] + mean_ov[1]],
            mode='lines',
            line=dict(color=c, width=3),
            showlegend=False
        ), row=1, col=2)
        
        # 3. Arrow Head
        fig.add_trace(go.Scatter(
            x=[mean_input[0] + mean_ov[0]],
            y=[mean_input[1] + mean_ov[1]],
            mode='markers',
            marker=dict(size=8, color=c, symbol='arrow-bar-up', angleref="previous"),
            showlegend=False
        ), row=1, col=2)

    fig.update_layout(
        title=f"L{layer_idx}H{head_idx} OV Circuit Analysis (Basis: Resid_Mid)",
        width=1100, height=600,
        template="plotly_white",
        xaxis_title="PC1 (Resid_Mid)", yaxis_title="PC2 (Resid_Mid)",
        xaxis2_title="PC1 (Resid_Mid)", yaxis2_title="PC2 (Resid_Mid)"
    )
    fig.show()


In [25]:
plot_real_ov_vectors(model, process, num_seqs=1000, seq_len=10, layer_idx=0, head_idx=0,start_pos=2)

In [ ]:
import torch
from sklearn.decomposition import PCA
import scipy.linalg
import numpy as np
import pandas as pd
import itertools

def measure_quadrilateral_overlap(model, process, num_seqs=1000, seq_len=10, layer_idx=0, head_idx=0, k=10):
    device = model.cfg.device
    
    if seq_len > model.cfg.n_ctx:
        seq_len = model.cfg.n_ctx

    if hasattr(process, 'generate_batch_gpu'):
        try:
            input_batch = process.generate_batch_gpu(batch_size=num_seqs, seq_len=seq_len, device=device)
        except TypeError:
             input_batch = process.generate_batch_gpu(batch_size=num_seqs, device=device)
   
    input_batch = input_batch[:, :seq_len]
    print(f"Running model with batch shape: {input_batch.shape}")
    
    _, cache = model.run_with_cache(input_batch)

    resid_pre = cache[f"blocks.{layer_idx}.hook_resid_pre"].reshape(-1, model.cfg.d_model).detach().cpu().float()
    
    resid_mid = cache[f"blocks.{layer_idx}.hook_resid_mid"].reshape(-1, model.cfg.d_model).detach().cpu().float()
    
    attn_out = cache[f"blocks.{layer_idx}.hook_attn_out"].reshape(-1, model.cfg.d_model).detach().cpu().float()
    
    v_hook = cache[f"blocks.{layer_idx}.attn.hook_v"][:, :, head_idx, :] # [batch, seq, d_head]
    W_O = model.blocks[layer_idx].attn.W_O[head_idx].detach().cpu()      # [d_head, d_model]
    
    ov_out = (v_hook.cpu() @ W_O).reshape(-1, model.cfg.d_model).float() # [batch*seq, d_model]

    spaces = {
        "Pre (Input)": resid_pre,
        "Mid (Total State)": resid_mid,
        "AttnOut (All Heads)": attn_out,
        "OV (Head Only)": ov_out
    }
    
    bases = {}
    for name, data in spaces.items():
        pca = PCA(n_components=k)
        pca.fit(data)
        bases[name] = pca.components_.T

    results = []
    
    space_names = list(spaces.keys())
    pairs = list(itertools.combinations(space_names, 2))
    
    print(f"\n=== Layer {layer_idx} Head {head_idx} Subspace Analysis (k={k}) ===")
    
    for name_a, name_b in pairs:
        U_a = bases[name_a]
        U_b = bases[name_b]
        
        #Subspace Overlap (0-1)
        # trace(A^T B B^T A) or norm(U_a^T U_b) / sqrt(k)
        interaction = np.dot(U_a.T, U_b)
        overlap = np.linalg.norm(interaction) / np.sqrt(k)
        
        #Canonical Angles
        angles_rad = scipy.linalg.subspace_angles(U_a, U_b)
        angles_deg = np.rad2deg(angles_rad)
        
        print(f"\nComparison: {name_a} vs {name_b}")
        print(f"  Overlap Score: {overlap:.4f}")
        
        results.append({
            "Space A": name_a,
            "Space B": name_b,
            "Overlap": overlap,
            "Min Angle": np.min(angles_deg),
            "Max Angle": np.max(angles_deg)
        })

    return pd.DataFrame(results)


In [27]:
df_results=measure_quadrilateral_overlap(model, process,num_seqs=10000, layer_idx=0, head_idx=0, k=2)

Running model with batch shape: torch.Size([10000, 10])

=== Layer 0 Head 0 Subspace Analysis (k=2) ===

Comparison: Pre (Input) vs Mid (Total State)
  Overlap Score: 0.4636

Comparison: Pre (Input) vs AttnOut (All Heads)
  Overlap Score: 0.1305

Comparison: Pre (Input) vs OV (Head Only)
  Overlap Score: 0.1215

Comparison: Mid (Total State) vs AttnOut (All Heads)
  Overlap Score: 0.9101

Comparison: Mid (Total State) vs OV (Head Only)
  Overlap Score: 0.8743

Comparison: AttnOut (All Heads) vs OV (Head Only)
  Overlap Score: 0.9680


In [28]:
print(df_results)

               Space A              Space B   Overlap  Min Angle  Max Angle
0          Pre (Input)    Mid (Total State)  0.463589  58.596450  66.553065
1          Pre (Input)  AttnOut (All Heads)  0.130475  79.483459  88.447203
2          Pre (Input)       OV (Head Only)  0.121485  80.260538  88.281627
3    Mid (Total State)  AttnOut (All Heads)  0.910147  22.042956  26.737581
4    Mid (Total State)       OV (Head Only)  0.874276  26.617094  31.342326
5  AttnOut (All Heads)       OV (Head Only)  0.968039   4.567669  20.220007


In [31]:
def plot_attention_decay(model,num_seqs,layer_idx,head_idx):
    device=torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
    if hasattr(process,'generate_batch_gpu'):
        input_batch=process.generate_batch_gpu(batch_size=num_seqs, seq_len=seq_len,device=device)
    _,cache=model.run_with_cache(input_batch)
    patterns = cache[f"blocks.{layer_idx}.attn.hook_pattern"][:, head_idx, :, :].detach().cpu()
    indices = torch.arange(seq_len).unsqueeze(0) - torch.arange(seq_len).unsqueeze(1)
    valid_mask = indices <= 0
    batch_indices = indices.unsqueeze(0).expand(num_seqs, -1, -1)
    valid_distances = batch_indices[valid_mask.unsqueeze(0).expand(num_seqs, -1, -1)].flatten()
    valid_attentions = patterns[:, valid_mask].flatten()
    df_attn = pd.DataFrame({
        'distance': valid_distances.numpy(),
        'attention': valid_attentions.numpy()
    })
    
    # Group by distance and compute mean
    df_mean = df_attn.groupby('distance')['attention'].mean().reset_index()
    fig = go.Figure()
    
    # Scatter of all points (optional, can be messy) or just mean line
    fig.add_trace(go.Scatter(
        x=df_mean['distance'],
        y=df_mean['attention'],
        mode='lines+markers',
        name='Actual Attention',
        line=dict(color='black', width=2),
    ))
    
    fig.update_layout(
        title="Figure 2C: Attention Pattern vs Relative Source Position",
        xaxis_title="Source Position Relative to Destination",
        yaxis_title="Attention Value",
        template="plotly_white",
        width=800, height=600
    )
    fig.show()


In [32]:
plot_attention_decay(model, num_seqs=50, layer_idx=0, head_idx=1)

RuntimeError: indices should be either on cpu or on the same device as the indexed tensor (cpu)

In [33]:
def plot_attention_decay(model, process, num_seqs, layer_idx, head_idx, seq_len=10, x_param=0.15):
    """
    Plots the average attention pattern vs distance and compares it to the 
    theoretical prediction: (1 - 3x)^|distance|.
    
    Args:
        x_param: The 'x' parameter from the Mess3 process (default 0.15 as per paper).
    """
    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    # Handle MPS for Mac
    if torch.backends.mps.is_available():
        device = torch.device("mps")

    if hasattr(process, "_ensure_gpu_tensors"):
         process._ensure_gpu_tensors(device)
    
    if hasattr(process, 'generate_batch_gpu'):
        input_batch = process.generate_batch_gpu(batch_size=num_seqs, seq_len=seq_len, device=device)
    else:
        input_batch = torch.tensor(
            [process.generate_process_history(total_length=seq_len).symbols for _ in range(num_seqs)],
            dtype=torch.long, device=device
        )    
    _, cache = model.run_with_cache(input_batch)
    
    patterns = cache[f"blocks.{layer_idx}.attn.hook_pattern"][:, head_idx, :, :].detach().cpu()
    
    # 4. Vectorized Distance Calculation
    # indices[i, j] = j - i (Source - Destination). e.g., 0 - 2 = -2
    indices = torch.arange(seq_len).unsqueeze(0) - torch.arange(seq_len).unsqueeze(1)
    
    valid_mask = indices <= 0
    batch_indices = indices.unsqueeze(0).expand(num_seqs, -1, -1)
    valid_distances = batch_indices[valid_mask.unsqueeze(0).expand(num_seqs, -1, -1)].flatten().abs()
    valid_attentions = patterns[:, valid_mask].flatten()
    
    df_attn = pd.DataFrame({
        'distance': valid_distances.numpy(),
        'attention': valid_attentions.numpy()
    })
    
    df_mean = df_attn.groupby('distance')['attention'].mean().reset_index()
    
    distances_theory = df_mean['distance'].values
    
    
    base = (1 - 3 * x_param)
    theory_values = base ** distances_theory
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=-df_mean['distance'], # Plot on negative x-axis (Source Relative to Dest)
        y=df_mean['attention'],
        mode='lines+markers',
        name='Actual Attention',
        line=dict(color='black', width=3),
        marker=dict(size=8)
    ))
    
    # Theoretical
    fig.add_trace(go.Scatter(
        x=-distances_theory,
        y=theory_values, # You might need to scale this if it doesn't match y-axis range
        mode='lines',
        name=f'Theory $(1 - 3x)^{{|n-1|}}$',
        line=dict(color='red', width=2, dash='dash')
    ))
    
    fig.update_layout(
        title=f"Figure 2C: Attention Decay (x={x_param})",
        xaxis_title="Source Position Relative to Destination",
        yaxis_title="Attention Value",
        template="plotly_white",
        width=800, height=600,
        legend=dict(x=0.05, y=0.95)
    )
    fig.show()

# Example Usage:
plot_attention_decay(model, process, num_seqs=100, layer_idx=0, head_idx=0, x_param=0.5)

RuntimeError: indices should be either on cpu or on the same device as the indexed tensor (cpu)

In [30]:
plot_attention_decay(model, process, num_seqs=100, layer_idx=0, head_idx=1, x_param=0.5)

RuntimeError: indices should be either on cpu or on the same device as the indexed tensor (cpu)

In [31]:
def plot_real_ov_vectors(model, process, num_seqs=50, seq_len=10,layer_idx=0, head_idx=0):
    # 1. Generate Batch Data (Vectorized)
    if hasattr(process, 'generate_batch_gpu'):
        input_batch = process.generate_batch_gpu(batch_size=num_seqs, seq_len=seq_len, device=device)
    # else:
    #     seqs = [process.generate_process_history(total_length=seq_len).symbols for _ in range(num_seqs)]
    #     input_batch = torch.tensor(seqs, dtype=torch.long, device=device)
        
    _, cache = model.run_with_cache(input_batch)
    
    #[Batch, seq_len, n_heads, d_head]
    v_batch = cache[f"blocks.{layer_idx}.attn.hook_v"][:, :, head_idx, :].cpu()

    W_O = model.blocks[layer_idx].attn.W_O[head_idx].detach().cpu() 
    if W_O.dim() > 2: W_O = W_O.squeeze()

    ov_updates_batch = v_batch @ W_O
    
    #[Batch * seq_len, d_model]
    all_vectors = ov_updates_batch.reshape(-1, ov_updates_batch.shape[-1])
    #flatten labels
    labels = input_batch.cpu().flatten().tolist()

    #token embedding
    static_embeddings = model.W_E.detach().cpu()

    #pca
    pca = PCA(n_components=2)
    pca.fit(all_vectors)
    
    ov_pca = pca.transform(all_vectors)
    emb_pca = pca.transform(static_embeddings)
    fig = go.Figure()
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

    for i in range(model.cfg.d_vocab):
        fig.add_trace(go.Scatter(
            x=[emb_pca[i, 0]], y=[emb_pca[i, 1]],
            mode='markers',
            marker=dict(size=10, color=colors[i], symbol='circle', line=dict(width=1, color='black')),
            name=f'Embedding {i}'
        ))
    
    for token_id in range(model.cfg.d_vocab):
        indices = [idx for idx, label in enumerate(labels) if label == token_id]
        if not indices: continue
            
        #mean ov vector for the token
        token_ov_vectors = ov_pca[indices]
        mean_vector = token_ov_vectors.mean(axis=0)
        
        # arrow from origin to mean vector
        fig.add_trace(go.Scatter(
            x=[0, mean_vector[0]], 
            y=[0, mean_vector[1]],
            mode='lines+markers',
            marker=dict(size=12, symbol='arrow-bar-up', angleref="previous", color=colors[token_id]),
            line=dict(width=4, color=colors[token_id]),
            name=f'OV Vector {token_id}'
        ))
        fig.add_trace(go.Scatter(
            x=token_ov_vectors[:, 0], 
            y=token_ov_vectors[:, 1], 
            mode='markers',
            marker=dict(size=4, color=colors[token_id], opacity=0.3),
            showlegend=False
        ))

    fig.update_layout(
        title=f" Embeddings vs OV Vectors (Layer {layer_idx}, Head {head_idx})",
        xaxis_title="PC1", yaxis_title="PC2",
        width=600, height=600,
        template="plotly_white",
        yaxis=dict(scaleanchor="x", scaleratio=1)
    )
    fig.show()


In [32]:
plot_real_ov_vectors(model, process, num_seqs=10, seq_len=10, layer_idx=0, head_idx=1)